# USA — Already-on-BFP APT Validation

## Purpose

Validate a set of **scoped APT ids** (USA) against the current layer `14533` and today's
Building Footprint (BFP), then emit the Data Correctness (DC) input format for the ones that
qualify.

## Steps

1. **Load snapshot for layer `14533`** and filter to USA address points. Dataset name: `usaAptDataset`.
2. **Load BFP** from catalog `preprocess_prod.bfp`. Dataset name: `usaBfpDataset`.

**Batch setup (run once):** build the file-independent join inputs (`latestForJoin`, `bfpForJoin`) and the output config (`outputDir`, `writeSingleCsv`).

**Per file (`processScopedFile`), run for every CSV in the folder via the driver loop (which lists `scopedFiles` and iterates):**

3. **Load the scoped CSV** as `scopedDS`.
4. **Join `scopedDS` with `latestForJoin`** on `Product_orbis_id` → `joinedDS` (+ `scopedNotExistDS`: ids not in the snapshot).
5. **Enrich `joinedDS`:** add `latest_vs_scoped_distance_in_meters` and `location_change`.
6. **Join with `bfpForJoin`:** add `is_intersects_with_bfp` and `bfp_geom`.
7. **Split** into `intersectingScope` (on a BFP) and `notIntersectingScope` (not on any BFP).
8. **Write** the DC-input CSV (`<file>-Already-On-BFP-DC-Input-format.csv`) and the not-on-BFP CSV (`<file>-not_on_bfp_scope.csv`); log per-file counts prefixed with the file name.

In [ ]:
%sql
DROP DATABASE IF EXISTS usa_alredy_on_bfp_validation CASCADE;
CREATE DATABASE IF NOT EXISTS usa_alredy_on_bfp_validation;

In [ ]:
%scala
// Shared constants used across the cells below (defined once here; values do not change between runs).
val LAYER_ID = 14533
val COUNTRY_ISO = "USA"
val NEW_DATABASE = "usa_alredy_on_bfp_validation"
val latestSnapshotTable = s"${NEW_DATABASE}.apt_snapshot_new"

In [ ]:
%scala
// The newest revision is resolved automatically via LoadSnapshotInfo.getLatestRevisionId.

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

val latestRevision = LoadSnapshotInfo.getLatestRevisionId(LAYER_ID)
println(s"Loading LATEST snapshot revision: ${latestRevision.get}")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(NEW_DATABASE)
new LoadFreshSnapshotData().run()

// Persist the loaded latest snapshot, keeping only APTs whose metadata:country tag = USA
val latestAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "metadata:country"
    && t.getField("value") === COUNTRY_ISO))
latestAptDS.write.format("delta").mode("overwrite").saveAsTable(latestSnapshotTable)

In [ ]:
%scala
println(s"LATEST snapshot written to: $latestSnapshotTable")

In [ ]:
%scala
// Read the persisted USA APT snapshot back from the catalog as usaAptDataset.
// Table: usa_alredy_on_bfp_validation.apt_snapshot_new (written in the previous cell).

import org.apache.spark.sql.{Dataset, Encoders}
import com.tomtom.orbis.io.spark.model.OrbisElement

val usaAptDataset: Dataset[OrbisElement] =
  spark.table(latestSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

display(usaAptDataset)

In [ ]:
%scala
println(s"usaAptDataset read from: $latestSnapshotTable, count: ${usaAptDataset.count()}")

In [ ]:
%scala
// Read the USA Building Footprint (BFP) polygons from preprocess_prod.bfp.

import org.apache.spark.sql.functions._
import org.apache.sedona.spark.SedonaContext

// Sedona is required for the ST_Intersects spatial join in the next cell
val sedona = SedonaContext.create(spark)

val usaBfpDataset = spark.table("preprocess_prod.bfp")
  .filter(col("licenseZone") === COUNTRY_ISO)
  .filter(col("wkt").isNotNull)
  .withColumn("bfp_geom", expr("ST_GeomFromWKT(wkt)"))

display(usaBfpDataset)

In [ ]:
%scala
println(s"USA BFP polygon count: ${usaBfpDataset.count()}")

In [ ]:
%scala
// Batch setup (1/3) - build the file-independent join inputs ONCE (reused for every scoped file).
import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.Id
import java.lang.{Long => JLong}

// Build the underscore-separated unsigned id: {layerId}_{unsignedHigh}_{unsignedLow}
def convertOrbisIdToProductId(orbisId: Id): String = {
  Seq(orbisId.layerId.getOrElse(LAYER_ID).toString,
      JLong.toUnsignedString(orbisId.high),
      JLong.toUnsignedString(orbisId.low)).mkString("_")
}
val convertOrbisIdToProductIdUDF = udf((orbisId: Id) => convertOrbisIdToProductId(orbisId))

// Latest snapshot reduced to join key + location + tags (file-independent -> cache & reuse per file).
val latestForJoin = usaAptDataset
  .withColumn("Product_orbis_id", convertOrbisIdToProductIdUDF(col("id")))
  .withColumnRenamed("lat", "latest_lat")
  .withColumnRenamed("lng", "latest_lon")
  .withColumnRenamed("tags", "latest_tags")
  .withColumn("latest_wkt", concat(lit("POINT("), col("latest_lon"), lit(" "), col("latest_lat"), lit(")")))
  .select("Product_orbis_id", "latest_lat", "latest_lon", "latest_wkt", "latest_tags")
  .cache()

// A spatial LEFT join is NOT optimized by Sedona -> disable broadcast so the inner ST_Intersects
// join is planned as a Sedona partitioned RangeJoin (avoids EXECUTOR_BROADCAST_JOIN_OOM).
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

// BFP polygons prepared once for the spatial join (bfp_geom_wkt carries the matched footprint WKT).
val bfpForJoin = usaBfpDataset
  .select(col("bfpId").as("bfp_id"), col("wkt").as("bfp_geom_wkt"), col("bfp_geom"))

println(s"latestForJoin rows: ${latestForJoin.count()}")

In [ ]:
%scala
// Batch setup (2/3) - output directory + single-CSV writer (shared by every file).
import org.apache.spark.sql.DataFrame

// >>> Set the output directory here <<<
val outputDir = "/mnt/aqua/data-correctness/correction-input/SEACO-6267-USA-Already-On-BFP-Scope/ReScoped_AlreadyOnBFP_Data/"

// Write a DataFrame as one named .csv file: Spark emits a folder of part files, so coalesce to a
// single partition, then promote that part file to the target name and clean up the temp dir.
// stripSuffix("/") keeps the path clean even when outputDir has a trailing slash.
def writeSingleCsv(df: DataFrame, fileName: String): Unit = {
  val base = outputDir.stripSuffix("/")
  val target = s"$base/$fileName"
  val tmpDir = s"$base/${fileName}_tmp"
  df.coalesce(1).write.option("header", "true").mode("overwrite").csv(tmpDir)
  val partFile = dbutils.fs.ls(tmpDir).filter(_.name.endsWith(".csv")).head.path
  dbutils.fs.rm(target)
  dbutils.fs.cp(partFile, target)
  dbutils.fs.rm(tmpDir, recurse = true)
  println(s"CSV written to: $target")
}

In [ ]:
%scala
// Per-file pipeline (Steps 3-8) - read one scoped CSV, join to the snapshot, compute distance /
// location_change, intersect with BFP, split into on-/not-on-BFP, write both CSVs, and log counts.
// Every print is prefixed with the scoped file name so batch logs are attributable per file.
// latestForJoin, bfpForJoin, outputDir and writeSingleCsv are the file-independent setup above.
import org.apache.spark.sql.functions._

def processScopedFile(scopedCsvPath: String): Unit = {
  val scopedFileName = scopedCsvPath.split("/").last
  val scopedFileBaseName = scopedFileName.stripSuffix(".csv")

  // Step 3 - load the scoped CSV (Target_Value_old_xy = "POINT (lon lat)")
  val scopedDS = spark.read.option("header", "true").csv(scopedCsvPath)
    .withColumn("scoped_lon",
      regexp_extract(col("Target_Value_old_xy"), """POINT \(([^ ]+) ([^)]+)\)""", 1).cast("double"))
    .withColumn("scoped_lat",
      regexp_extract(col("Target_Value_old_xy"), """POINT \(([^ ]+) ([^)]+)\)""", 2).cast("double"))
    .withColumn("scoped_wkt", concat(lit("POINT("), col("scoped_lon"), lit(" "), col("scoped_lat"), lit(")")))

  // Step 4 - join to the latest snapshot on Product_orbis_id
  val joinedDS = scopedDS.join(latestForJoin, Seq("Product_orbis_id"), "left")
  val scopedNotExistDS = scopedDS.join(latestForJoin, Seq("Product_orbis_id"), "left_anti")

  // Step 5 - Haversine distance + location_change
  val EARTH_RADIUS_M = 6371008.8
  val dLat = radians(col("latest_lat") - col("scoped_lat"))
  val dLon = radians(col("latest_lon") - col("scoped_lon"))
  val haversine =
    pow(sin(dLat / 2), 2) +
    cos(radians(col("scoped_lat"))) * cos(radians(col("latest_lat"))) * pow(sin(dLon / 2), 2)
  val distanceMeters = lit(2 * EARTH_RADIUS_M) * asin(sqrt(haversine))
  val joinedWithDistance = joinedDS
    .withColumn("latest_vs_scoped_distance_in_meters",
      when(col("latest_lat").isNotNull && col("scoped_lat").isNotNull, distanceMeters))
    .withColumn("location_change",
      when(col("latest_lat").isNull || col("scoped_lat").isNull, lit(null).cast("boolean"))
        .otherwise(!(col("latest_lat") === col("scoped_lat") && col("latest_lon") === col("scoped_lon"))))

  // Step 6 - BFP intersection of the latest point (inner Sedona RangeJoin, then left join back)
  val aptPoints = joinedWithDistance
    .filter(col("latest_lat").isNotNull && col("latest_lon").isNotNull)
    .withColumn("apt_geom", expr("ST_GeomFromText(concat('POINT(', latest_lon, ' ', latest_lat, ')'))"))
  val matched = aptPoints.alias("apt")
    .join(bfpForJoin.alias("bfp"), expr("ST_Intersects(bfp.bfp_geom, apt.apt_geom)"))
    .select(
      col("apt.Product_orbis_id").as("Product_orbis_id"),
      col("bfp.bfp_id").as("bfp_id"),
      col("bfp.bfp_geom_wkt").as("bfp_geom"))
    .dropDuplicates("Product_orbis_id")
  val joinedWithBfp = joinedWithDistance
    .join(matched, Seq("Product_orbis_id"), "left")
    .withColumn("is_intersects_with_bfp", col("bfp_id").isNotNull)

  // Step 7 - split into on-BFP (validated) and not-on-BFP
  val intersectingScope = joinedWithBfp.filter(col("is_intersects_with_bfp") === true).cache()
  val notIntersectingScope = joinedWithBfp.filter(col("is_intersects_with_bfp") === false)

  // Step 8 - DC-input format + not-on-BFP scope, write both CSVs (names derived from the input file)
  // For Relocation_Type = ALREADY_ON_CORRECT_BFP, Product_Address_component and target_value are left blank.
  val isAlreadyOnCorrectBfp = col("Relocation_Type") === "ALREADY_ON_CORRECT_BFP"
  val dcInputScope = intersectingScope
    .withColumn("Product_Address_component",
      when(isAlreadyOnCorrectBfp, lit("")).otherwise(lit("metadata:apa:previous_location")))
    .withColumn("target_value",
      when(isAlreadyOnCorrectBfp, lit("")).otherwise(col("Target_Value_old_xy")))
    .withColumn("Orbis_Y", regexp_extract(col("latest_wkt"), """POINT\(([^ ]+) ([^)]+)\)""", 1))
    .withColumn("Orbis_X", regexp_extract(col("latest_wkt"), """POINT\(([^ ]+) ([^)]+)\)""", 2))
    .withColumn("Relocation_Type", col("Relocation_Type"))
    .select("Product_orbis_id", "Product_Address_component", "target_value",
            "Orbis_X", "Orbis_Y", "Relocation_Type")
  // Drop the tags column (array<struct>, not CSV-serializable) before writing.
  val notOnBfpScope = notIntersectingScope.drop("latest_tags")

  writeSingleCsv(dcInputScope, s"$scopedFileBaseName-Already-On-BFP-DC-Input-format.csv")
  writeSingleCsv(notOnBfpScope, s"$scopedFileBaseName-not_on_bfp_scope.csv")

  // Per-file summary log
  println(s"[$scopedFileName] scopedDS: ${scopedDS.count()}")
  println(s"[$scopedFileName] matched in snapshot: ${joinedDS.filter(col("latest_wkt").isNotNull).count()}, " +
    s"not in snapshot (scopedNotExistDS): ${scopedNotExistDS.count()}")
  println(s"[$scopedFileName] joinedWithDistance rows: ${joinedWithDistance.count()}, " +
    s"location_change=true: ${joinedWithDistance.filter(col("location_change") === true).count()}")
  println(s"[$scopedFileName] joinedWithBfp: ${joinedWithBfp.count()}, " +
    s"intersecting a BFP (intersectingScope): ${intersectingScope.count()}, " +
    s"not intersecting (notIntersectingScope): ${notIntersectingScope.count()}")
  println(s"[$scopedFileName] DC input scope count: ${dcInputScope.count()}")

  intersectingScope.unpersist()
}

In [ ]:
%scala
// Driver - list every scoped CSV in the folder and run the per-file pipeline for each.
val scopedFolder = "/mnt/aqua/data-correctness/correction-input/SEACO-6267-USA-Already-On-BFP-Scope/ALREADY_ON_CORRECT_BFP/"
val scopedFiles = dbutils.fs.ls(scopedFolder).filter(_.name.endsWith(".csv")).map(_.path)

println(s"Found ${scopedFiles.length} scoped CSV file(s) to process")

scopedFiles.foreach { path =>
  println(s"==== Processing ${path.split("/").last} ====")
  processScopedFile(path)
}
println("==== Done processing all scoped files ====")